<a href="https://colab.research.google.com/github/OlajideFemi/Carbon-Footprint/blob/main/Record_Linkage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Fuzzy Matching

Levenshtein distance

- Standardisation


In [1]:
!pip install thefuzz python-Levenshtein

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 44.6 MB/s eta 0:00:00


In [2]:
!pip install openpyxl

In [3]:
!pip install tqdm

In [ ]:
!pip install

In [ ]:
# ============================================================
# Matching
# ============================================================

import pandas as pd
import re
import os
import time
from thefuzz import process, fuzz
from tqdm.auto import tqdm
from google.colab import drive

# --- 1. Mount Google Drive ---
# This allows the script to see "03_Sandbox" folder
drive.mount('/content/drive')

# --- 2. Setup Paths ---
# Adjust the BASE_PATH if folder is inside "My Drive"
BASE_PATH = "/content/drive/MyDrive/03_Sandbox/Grants_Analysis"
#https://www.gov.uk/government/statistics/government-grants-statistics-2024-to-2025
GRANTS_FILE = f"{BASE_PATH}/Government_Grants_Register_2024_to_2025.xlsx"
GRANTS_SHEET = "#Awards"
CH_MASTER_FILE = f"{BASE_PATH}/BasicCompanyDataAsOneFile-2026-05-01.csv"

# Ensure the output directory exists
os.makedirs(BASE_PATH, exist_ok=True)

OUTPUT_CLEANED_RECIPIENTS = f"{BASE_PATH}/Grant_Recipients_Cleaned.csv"
OUTPUT_ALL_MATCHES = f"{BASE_PATH}/All_CH_Matches.csv"
OUTPUT_BEST_MATCHES = f"{BASE_PATH}/Best_CH_Matches.csv"
OUTPUT_HIGH_CONFIDENCE = f"{BASE_PATH}/High_Confidence_CH_Matches.csv"
OUTPUT_REVIEW = f"{BASE_PATH}/Review_CH_Matches.csv"
OUTPUT_UNMATCHED = f"{BASE_PATH}/Unmatched_Grant_Recipients.csv"

# --- 3. Settings ---
CHUNKSIZE = 100000
FUZZY_THRESHOLD = 85
HIGH_CONFIDENCE_THRESHOLD = 92
REVIEW_THRESHOLD = 85
MAX_CANDIDATES_PER_BLOCK = 50000

CH_COLS = ["CompanyName", " CompanyNumber", "RegAddress.AddressLine1",
           "RegAddress.PostTown", "RegAddress.PostCode", "CompanyStatus"]

# --- 4. Logic Functions ---
def clean_name(name):
    if pd.isna(name): return ""
    name = str(name).upper().strip()
    name = name.replace("&", " AND ").replace("+", " AND ").replace("'", "").replace("’", "")
    name = re.sub(r"[^A-Z0-9\s]", " ", name)
    remove_words = {"LIMITED", "LTD", "PLC", "LLP", "CIC", "COMPANY", "CO", "INC",
                    "CORPORATION", "CORP", "GROUP", "HOLDINGS", "HOLDING", "UK",
                    "GB", "GREAT", "BRITAIN", "THE"}
    words = [w for w in name.split() if w not in remove_words]
    return re.sub(r"\s+", " ", " ".join(words)).strip()

def first_word_key(name):
    return name.split()[0] if name else ""

def classify_match(score):
    if score == 100: return "Exact"
    elif score >= HIGH_CONFIDENCE_THRESHOLD: return "High confidence"
    elif score >= REVIEW_THRESHOLD: return "Review"
    else: return "Low confidence"

# --- 5. Execution Pipeline ---
print(" Loading grants data...")
try:
    df = pd.read_excel(GRANTS_FILE, sheet_name=GRANTS_SHEET)
except Exception as e:
    print(f" Error loading file. Check if path is correct: {GRANTS_FILE}")
    raise e

df_unique = df.drop_duplicates(subset="Recipient Org:Name", keep="first").copy()
df_unique["Match_Name"] = df_unique["Recipient Org:Name"].apply(clean_name)
df_unique = df_unique[df_unique["Match_Name"] != ""].copy()
df_unique = df_unique.drop_duplicates(subset="Match_Name", keep="first").copy()
df_unique["block_first_word"] = df_unique["Match_Name"].apply(first_word_key)
df_unique["recipient_id"] = range(1, len(df_unique) + 1)
df_unique.to_csv(OUTPUT_CLEANED_RECIPIENTS, index=False)

recipient_name_set = set(df_unique["Match_Name"].tolist())
matched_recipient_ids = set()
all_matches = []

print(f" Unique cleaned recipients: {len(df_unique):,}")

# --- 6. Process Companies House ---
print("\n Processing Companies House data in chunks...")
start_time = time.time()

ch_iter = pd.read_csv(CH_MASTER_FILE, usecols=CH_COLS, dtype={" CompanyNumber": str},
                      chunksize=CHUNKSIZE, low_memory=False)

for chunk_index, chunk in enumerate(ch_iter, start=1):
    chunk_start = time.time()
    chunk["Match_Name_CH"] = chunk["CompanyName"].apply(clean_name)
    chunk = chunk[chunk["Match_Name_CH"] != ""].copy()
    if chunk.empty: continue

    chunk["block_first_word"] = chunk["Match_Name_CH"].apply(first_word_key)

    # 9A. Exact Matches
    exact_chunk = chunk[chunk["Match_Name_CH"].isin(recipient_name_set)].copy()
    if not exact_chunk.empty:
        exact_merged = exact_chunk.merge(
            df_unique[["recipient_id", "Recipient Org:Name", "Match_Name"]],
            left_on="Match_Name_CH", right_on="Match_Name", how="inner"
        )
        for _, row in exact_merged.iterrows():
            all_matches.append({
                "recipient_id": row["recipient_id"], "Recipient_Org_Name": row["Recipient Org:Name"],
                "Recipient_Match_Name": row["Match_Name"], "Best_CH_Match": row["CompanyName"],
                "Match_Score": 100, "Match_Type": "Exact", "Confidence": "Exact",
                "CompanyNumber": row.get(" CompanyNumber", ""), "Chunk_Index": chunk_index
            })
            matched_recipient_ids.add(row["recipient_id"])

    # 9B. Fuzzy Blocked Matches
    remaining_recipients = df_unique[~df_unique["recipient_id"].isin(matched_recipient_ids)].copy()
    if remaining_recipients.empty: break

    ch_groups = {k: g for k, g in chunk.groupby("block_first_word")}

    for block_key, recipient_group in remaining_recipients.groupby("block_first_word"):
        if block_key == "" or block_key not in ch_groups: continue

        candidates = ch_groups[block_key]
        if len(candidates) > MAX_CANDIDATES_PER_BLOCK: candidates = candidates.head(MAX_CANDIDATES_PER_BLOCK)

        ch_names = candidates["Match_Name_CH"].unique().tolist()

        for _, rec_row in recipient_group.iterrows():
            if rec_row["recipient_id"] in matched_recipient_ids: continue

            res = process.extractOne(rec_row["Match_Name"], ch_names,
                                     scorer=fuzz.token_set_ratio, score_cutoff=FUZZY_THRESHOLD)
            if res:
                best_comp = candidates[candidates["Match_Name_CH"] == res[0]].iloc[0]
                all_matches.append({
                    "recipient_id": rec_row["recipient_id"], "Recipient_Org_Name": rec_row["Recipient Org:Name"],
                    "Recipient_Match_Name": rec_row["Match_Name"], "Best_CH_Match": best_comp["CompanyName"],
                    "Match_Score": res[1], "Match_Type": "Fuzzy blocked", "Confidence": classify_match(res[1]),
                    "CompanyNumber": best_comp.get(" CompanyNumber", ""), "Chunk_Index": chunk_index
                })

    print(f"Chunk {chunk_index} done | Matches: {len(all_matches)} | Time: {time.time()-start_time:.1f}s")

# --- 7. Finalise ---
if all_matches:
    res_df = pd.DataFrame(all_matches).sort_values(["recipient_id", "Match_Score"], ascending=[True, False])
    best = res_df.drop_duplicates(subset="recipient_id", keep="first")
    best.to_csv(OUTPUT_BEST_MATCHES, index=False)
    print(f"\n Done! Match Rate: {len(best)/len(df_unique):.1%}")
else:
    print("No matches found.")